# 2.1.2 LSTM ablation study

**Owner:** Winston Ho Rui Wei  
**Reference model:** LSTM with target-aware attention  
**Initial run:** `20261104_014914_seed42`  
**Study:** `ablation_v1`

This study examines which components support personalised article retrieval and
whether the neural model adds value beyond article freshness. The working
shortlist contains 100 articles. The initial experiment exports an additional
100 articles as a buffer for later evaluation and business reranking.

The completed comparisons cover recency and popularity baselines, history
encoding, target-aware attention, article representations, seed variation and an
explicit freshness adjustment. Saved outputs are retained from the completed
study. Sections 13–15 reload the stored reports without training or scoring.

## Evaluation design

All catalogue comparisons use the same 5,000 validation requests and the full
time-eligible catalogue. The 2,000 requests reserved for model or blend selection
are excluded from this reporting sample. This is development analysis; the
labelled project holdout is not evaluated.

The primary metric is macro known-click Recall@100. Observed-click nDCG@100
measures ordering within the shortlist. Recorded-slate AUC, nDCG@5 and MRR are
diagnostics over shown candidates, not substitutes for catalogue retrieval
metrics. Unshown outcomes remain unknown.

Trained controls reuse the initial run's chosen hyperparameters and receive
four epochs at a constant learning rate. The full model, final-state control and
attention-without-LSTM control each use seeds 42, 43 and 44. Other trained
controls use seed 42. These are comparisons under a common training budget,
not independently tuned optima for each architecture.

The study stores metrics and per-request results under
`DATA_ROOT/Experiments/2.1_lstm/ablation/ablation_v1/`.
It does not replace the initial run's all-request prediction exports.

### Comparison and interpretation

- Compare each trained control with the full model using matching seeds.
  Compare the saved-model history permutation with the saved model, and the
  recency-candidate rankers with newest-first.
- Paired 95% bootstrap intervals resample users, retaining their requests
  together. Multi-seed comparisons average the seed-matched per-request
  differences. These intervals describe the sampled users conditional on the
  fitted models; three seeds do not fully characterise training uncertainty.
- The original decision rule also required an absolute difference larger than
  the full model's maximum-minus-minimum seed spread. This is a conservative
  study heuristic, not a formal significance threshold. The blend uses its own
  single-seed spread, as documented in Section 12.
- Compare neural retrieval with newest-first before attributing business value
  to an architectural component. A gain over another neural control does not
  establish a gain over the baseline or an increase in real-world click rate.
- Removing vectors and availability flags together tests overall dependence on
  those inputs. It does not isolate flag effects or establish target leakage.
- History buckets describe the user's complete supplied history; the model
  consumes at most its latest 50 events. Attention weights and perturbation
  results are model diagnostics, not causal explanations.

## 1. Setup

The experiments were completed on an L4 GPU. To reproduce model comparisons,
upload `lstm_data.py`, `lstm_model.py` and `ablation_variants.py` beside this
notebook, select a GPU runtime and run this setup before the experiment sections.
The helpers remain in the same folder as the initial experiment.

Data and saved runs are accessed through the shared Drive shortcut at
`/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder`.
Setup reuses the initial model's fitted transformations and article inputs.
It loads training and validation data, but not project-test behaviour labels.

To review existing results in a fresh CPU runtime, run Section 13, followed by
Sections 14 and 15. Those sections need only Drive access and pandas; the model
setup and experiment sections are not required.

In [ ]:
from pathlib import Path

EXPERIMENT_ID = "2.1_lstm"
OWNER = "Winston Ho Rui Wei"
DATA_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder")
MAIN_RUN_ID = "20261104_014914_seed42"   # tuned main run; hyperparameters and saved model load from it
RUN_ID = "ablation_v1"                   # same ID = resume; change RUN_ID whenever the protocol changes
SEEDS = (42, 43, 44)                     # full model and group B; other arms use SEEDS[0]
EPOCHS = 4                               # fixed budget, constant learning rate, no early stopping

SAMPLE_SEED = 42                         # request-sampling seed (independent of model seeds)
EXCLUSION_REQUESTS = 2000                # final rerun's selection sample; never reported on
SELECTION_REQUESTS = 100                 # main run's selection sample (prefix check only)
CATALOGUE_REQUESTS = 5000
CATALOGUE_SAMPLE_LABEL = "validation_ablation_catalogue"   # BLAKE2b salt replacing the split name
K = 100                                  # selection cutoff (Recall@100, nDCG@100)
N_BOOT, BOOT_SEED = 2000, 0
CATALOGUE_CHUNK = 32768

In [ ]:
import sys, json, time, importlib.util
from dataclasses import asdict, fields, replace

if "google.colab" in sys.modules or Path("/content").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
    CODE_DIR = Path("/content")
else:
    CODE_DIR = Path.cwd()
sys.path.insert(0, str(CODE_DIR))
for name in ("lstm_data", "lstm_model", "ablation_variants"):
    sys.modules.pop(name, None)
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import matplotlib.pyplot as plt
import lstm_data as D
import lstm_model as M
import ablation_variants as V

MAIN_RUN_DIR = DATA_ROOT / "Experiments" / EXPERIMENT_ID / MAIN_RUN_ID
ABLATION_DIR = DATA_ROOT / "Experiments" / EXPERIMENT_ID / "ablation" / RUN_ID
ARMS_DIR = ABLATION_DIR / "arms"
ARMS_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAIN_SEED = SEEDS[0]
ENSEMBLE_LABEL = "_".join(map(str, SEEDS))

def log(*parts):
    print(*parts, flush=True)

RUN_CONFIG = json.loads((MAIN_RUN_DIR / "run_config.json").read_text())
MODEL_CFG = M.ModelConfig.from_json(json.loads((MAIN_RUN_DIR / "model" / "model_config.json").read_text()))
PREP = json.loads((MAIN_RUN_DIR / "model" / "preprocessing.json").read_text())
MAIN_HISTORY = json.loads((MAIN_RUN_DIR / "training_history.json").read_text())
TRAIN_CFG = M.TrainConfig(**RUN_CONFIG["train_config"])
log("Ablation run", RUN_ID, "| device", DEVICE, "| results:", ABLATION_DIR)
log("Main run by epoch: Recall@100", [r["known_click_recall@100"] for r in MAIN_HISTORY],
    "| learning rate", [f"{r['learning_rate']:.2e}" for r in MAIN_HISTORY])
log("Tuned model:", {k: v for k, v in MODEL_CFG.to_json().items()
                     if k in ("variant", "lstm_hidden", "attention_dim", "article_dim", "dropout", "text_dropout")})
log("Tuned training:", {k: getattr(TRAIN_CFG, k) for k in
                        ("learning_rate", "weight_decay", "history_dropout", "batch_impressions", "grad_clip")})

Mounted at /content/drive
Ablation run ablation_v1 | device cuda | results: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.1_lstm/ablation/ablation_v1
Main run by epoch: Recall@100 [0.55, 0.63, 0.73, 0.735, 0.715, 0.665] | learning rate ['1.53e-03', '1.53e-03', '1.53e-03', '1.53e-03', '1.53e-03', '7.66e-04']
Tuned model: {'variant': 'attention', 'article_dim': 128, 'lstm_hidden': 128, 'attention_dim': 32, 'dropout': 0.2803345035229627, 'text_dropout': 0.2}
Tuned training: {'learning_rate': 0.0015317709628184257, 'weight_decay': 1.1527987128232402e-06, 'history_dropout': 0.29097295564859826, 'batch_impressions': 256, 'grad_clip': 1.0}


In [ ]:
PATHS, _ = D.resolve_paths(DATA_ROOT, required=["features_train", "features_validation", "history_train",
                                                "history_validation", "behaviors_train", "behaviors_validation"])
FRAMES, OFFSETS, SPLIT_START = {}, {}, {}
for split in ("train", "validation"):               # the test split is never loaded
    frame, info = D.load_candidate_table(PATHS[f"features_{split}"], split)
    FRAMES[split], OFFSETS[split] = frame, info["offsets"]
    SPLIT_START[split] = pd.Timestamp(frame["impression_time"].min())
HISTORIES = {s: D.build_history(PATHS[f"history_{s}"], SPLIT_START[s], PREP["history_max_len"])
             for s in FRAMES}

with np.load(MAIN_RUN_DIR / "model" / "article_inputs.npz", allow_pickle=False) as saved:
    article_fields = {f.name: saved[f.name] for f in fields(D.ArticleTable)}
article_fields["numeric_names"] = article_fields["numeric_names"].tolist()
ARTICLES = D.ArticleTable(**article_fields)
NUMERIC_NAMES = list(ARTICLES.numeric_names)

ARRAYS = {s: D.build_split_arrays(s, FRAMES[s], OFFSETS[s], HISTORIES[s], ARTICLES,
                                  PREP["numeric_stats"], PREP["device_columns"]) for s in FRAMES}
VAL = ARRAYS["validation"]
EVENT_NAMES = list(VAL.event_names)
INDEX = D.build_catalogue_index("all", "union", ARTICLES.article_ids[1:], ARTICLES)
CLICKS_VAL = D.clicked_eligibility(VAL, INDEX, ARTICLES)
CLICK_LOOKUP = D.clicks_by_request(CLICKS_VAL)
AGE_STATS = PREP["numeric_stats"]["log_candidate_age_h"]

EXCLUSION, EXCLUSION_BIDS = D.select_requests(VAL.user_id, VAL.impression_id, VAL.imp_time_us,
                                              "validation", SAMPLE_SEED, EXCLUSION_REQUESTS)
SELECTION, SELECTION_BIDS = D.select_requests(VAL.user_id, VAL.impression_id, VAL.imp_time_us,
                                              "validation", SAMPLE_SEED, SELECTION_REQUESTS)
main_selection = pd.read_parquet(MAIN_RUN_DIR / "selection" / "validation_requests.parquet")["behaviour_id"].tolist()
REMAINING = np.setdiff1d(np.arange(VAL.n_impressions), EXCLUSION)
CAT_SAMPLE, CAT_BIDS = V.hash_sample(VAL.user_id, VAL.impression_id, VAL.imp_time_us,
                                     CATALOGUE_SAMPLE_LABEL, SAMPLE_SEED, CATALOGUE_REQUESTS, among=REMAINING)
SLATE_SAMPLE = REMAINING
SLATE_BIDS = D.behaviour_ids(VAL.user_id[SLATE_SAMPLE], VAL.impression_id[SLATE_SAMPLE]).tolist()
CAT_USERS, SLATE_USERS = VAL.user_id[CAT_SAMPLE], VAL.user_id[SLATE_SAMPLE]
SLATE_ARRAYS = D.subset_split_arrays(VAL, SLATE_SAMPLE)
lo, hi = INDEX.bounds(VAL.imp_time_us[CAT_SAMPLE])
CAT_PAIRS = int((hi - lo).sum())
log("Requests/candidates:", {s: (a.n_impressions, a.n_candidates) for s, a in ARRAYS.items()})
log("Selection sample inside the exclusion set:", set(SELECTION_BIDS) <= set(EXCLUSION_BIDS),
    "| equals the main run's saved selection IDs:", main_selection == SELECTION_BIDS)
log("Catalogue sample:", len(CAT_BIDS), "requests,", CAT_PAIRS, "eligible pairs | recorded slates:",
    len(SLATE_BIDS), "requests,", SLATE_ARRAYS.n_candidates, "candidates")

Requests/candidates: {'train': (232887, 2585747), 'validation': (29110, 344666)}
Selection sample inside the exclusion set: True | equals the main run's saved selection IDs: True
Catalogue sample: 5000 requests, 619774358 eligible pairs | recorded slates: 27110 requests, 321310 candidates


In [ ]:
S3, S1 = list(SEEDS), [MAIN_SEED]
ARMS = {   # trained arms: each changes one thing relative to "full"; a section adds any input only it needs
    "full": dict(tier=1, group="Seeds", seeds=S3, change="none: tuned main model under the fixed 4-epoch budget"),
    "last_state": dict(tier=1, group="B", seeds=S3, model=dict(variant="last_state"),
                       change="history summary = final LSTM state (no target-aware attention)"),
    "attention_no_lstm": dict(tier=1, group="B", seeds=S3, model=dict(variant="attention_no_lstm"),
                              change="attention over per-event inputs (no LSTM)"),
    "metadata_only": dict(tier=1, group="D", seeds=S1,
                          ablation=dict(use_text_vectors=False, use_image_vectors=False,
                                        drop_article_numeric=V.VECTOR_FLAG_COLUMNS),
                          change="both vector blocks and all three availability flags removed"),
    "no_history": dict(tier=1, group="D", seeds=S1, data="no_history",
                       change="history removed: empty-history path and context for every request"),
    "no_text": dict(tier=1, group="D", seeds=S1, ablation=dict(use_text_vectors=False),
                    change="text vector block removed (has_text flag kept)"),
    "no_image": dict(tier=1, group="D", seeds=S1, ablation=dict(use_image_vectors=False),
                     change="image vector block removed (has_image flag kept)"),
}
COMPARATORS = {   # no training; "reference" = full seeds to pair with, or another arm's name
    "most_recent": dict(tier=1, group="A", seeds=S1, reference=S1,
                        change="no model: eligible catalogue by publication time, newest first"),
    "popularity_24h": dict(tier=1, group="A", seeds=S1, reference=S1,
                           change="no model: clicks in the trailing 24 h, ties by recency"),
    "main_model": dict(tier=1, group="A", seeds=S1, reference=S1,
                       change="saved main-run model (no retraining) on the full catalogue"),
    **{f"rerank_{n}": dict(tier=1, group="A", seeds=S1, reference="most_recent",
                           change=f"newest {n} eligible articles, reranked by the main-run model")
       for n in (200, 500, 1000)},
    "logistic_regression": dict(tier=1, group="A", seeds=S1, reference=S1,
                                change="Baseline Testing logistic regression; recorded slates only"),
    "permute_eval": dict(tier=1, group="C", seeds=S1, reference="main_model",
                         change="main-run model, validation histories permuted at evaluation"),
    "ensemble": dict(tier=1, group="Seeds", seeds=[ENSEMBLE_LABEL], reference=S3,
                     change="mean logit of the three full-model seeds"),
}

ARTICLE_TENSORS = M.article_table_tensors(ARTICLES, DEVICE)
BASE_TENSORS = {"train": M.SplitTensors(ARRAYS["train"], DEVICE), "val": M.SplitTensors(VAL, DEVICE),
                "slates": M.SplitTensors(SLATE_ARRAYS, DEVICE)}

In [ ]:
def arm_dir(name, seed):
    return ARMS_DIR / f"{name}_seed{seed}"

def done(name, seed):
    return (arm_dir(name, seed) / "metrics.json").is_file()

def cuda_sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def free_gpu():
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

def latency(catalogue_summary):
    seconds = catalogue_summary["scoring_seconds"]
    return {"catalogue_ms_per_request": 1e3 * seconds / len(CAT_SAMPLE),
            "catalogue_ms_per_1k_candidates": 1e6 * seconds / CAT_PAIRS}

def save_run(name, seed, metrics, catalogue=None, slates=None):
    out = arm_dir(name, seed)
    out.mkdir(parents=True, exist_ok=True)
    if catalogue is not None:
        V.catalogue_frame(CAT_SAMPLE, CAT_BIDS, CAT_USERS, catalogue).to_parquet(out / "catalogue.parquet", index=False)
    if slates is not None:
        V.slate_frame(SLATE_SAMPLE, SLATE_BIDS, SLATE_USERS, slates).to_parquet(out / "slates.parquet", index=False)
    D.write_json({"arm": name, "seed": seed, **metrics}, out / "metrics.json")   # last: marks completion
    log(f"{name} seed {seed} saved | Recall@100 {(metrics.get('catalogue') or {}).get('known_click_recall@100')} "
        f"| slate AUC {(metrics.get('recorded_slates') or {}).get('mean_impression_auc')}")

def evaluate_catalogue(model, val_tensors, members=None):
    models = members or [model]
    t0 = time.time()
    scorers = [M.CatalogueScorer(m, val_tensors, INDEX, AGE_STATS, chunk=CATALOGUE_CHUNK, requests=CAT_SAMPLE)
               for m in models]
    cuda_sync()
    setup = time.time() - t0
    if members:
        topk, tie_fn = V.EnsembleTopK(scorers).topk, None
    else:
        topk, tie_fn = scorers[0].topk, (lambda: scorers[0].last_tie_at_cutoff)
    cat, cat_per = V.evaluate_catalogue(topk, CAT_SAMPLE, CLICKS_VAL, K, tie_fn=tie_fn, sync=cuda_sync)
    return cat, cat_per, {"scorer_setup_seconds": setup, **latency(cat)}

def evaluate_slates(models, slate_tensors):
    t0 = time.time()
    scores = np.mean([M.predict_logits(m, slate_tensors, batch_size=TRAIN_CFG.eval_batch_impressions)
                      for m in models], axis=0)
    slate, slate_per = V.slate_metrics(SLATE_ARRAYS.offsets, SLATE_ARRAYS.labels, scores)
    return slate, slate_per, time.time() - t0

def evaluate_model(model, val_tensors, slate_tensors, members=None):
    """Catalogue sample (exact full-pool top 100) and recorded-slate sample."""
    cat, cat_per, timing = evaluate_catalogue(model, val_tensors, members)
    slate, slate_per, timing["slate_scoring_seconds"] = evaluate_slates(members or [model], slate_tensors)
    free_gpu()
    return cat, cat_per, slate, slate_per, timing

def load_main_model():
    """The saved main-run model, reloaded as experiment.ipynb's reload check does (bundled code + config)."""
    spec = importlib.util.spec_from_file_location("saved_lstm_model", MAIN_RUN_DIR / "model" / "code" / "lstm_model.py")
    BM = importlib.util.module_from_spec(spec)
    sys.modules["saved_lstm_model"] = BM
    spec.loader.exec_module(BM)
    cfg = BM.ModelConfig.from_json(json.loads((MAIN_RUN_DIR / "model" / "model_config.json").read_text()))
    model = BM.build_model(cfg, ARTICLE_TENSORS, DEVICE)
    model.load_state_dict(torch.load(MAIN_RUN_DIR / "model" / "model_state.pt", map_location=DEVICE, weights_only=True))
    return model.eval()

def arm_arrays(spec):
    """(train, validation) SplitArrays for a trained arm."""
    data, pair = spec.get("data"), (ARRAYS["train"], VAL)
    if data == "no_history":
        pair = tuple(V.remove_history(a, PREP["numeric_stats"]) for a in pair)
    elif data in ("history_10", "history_20"):
        pair = tuple(V.truncate_history(a, int(data.split("_")[1]), EVENT_NAMES) for a in pair)
    elif data == "shuffled":
        pair = tuple(D.permute_history(a, spec["permute_seed"]) for a in pair)
    train, val = pair
    if spec.get("train_requests") is not None:
        train = D.subset_split_arrays(train, spec["train_requests"])
    return train, val

def arm_model(name, seed):
    spec = ARMS[name]
    cfg = replace(MODEL_CFG, **spec.get("model", {}))
    acfg = V.AblationConfig(**spec.get("ablation", {}))
    M.set_seed(seed)
    model = V.build_arm_model(cfg, acfg, spec.get("article_tensors", ARTICLE_TENSORS), DEVICE,
                              NUMERIC_NAMES, EVENT_NAMES)
    return model, cfg, acfg

def load_arm_model(name, seed):
    model, _, _ = arm_model(name, seed)
    model.load_state_dict(torch.load(arm_dir(name, seed) / "model_state.pt", map_location=DEVICE, weights_only=True))
    return model.eval()

def selection_check(model):
    """Reproducibility only: Recall@100 on the main run's selection sample (inside the exclusion set)."""
    scorer = M.CatalogueScorer(model, BASE_TENSORS["val"], INDEX, AGE_STATS, chunk=CATALOGUE_CHUNK, requests=SELECTION)
    shortlists, _ = M.catalogue_shortlists(scorer, SELECTION, K)
    return {"sample": "main-run 100-request selection sample (exclusion set; not a reported result)",
            "recall@100": D.known_click_recall(shortlists, CLICKS_VAL, (K,))["known_click_recall@100"],
            "main_run_epoch4_recall@100": MAIN_HISTORY[EPOCHS - 1]["known_click_recall@100"]}

def run_trained_arm(name, seed):
    """Skip if metrics.json exists; else reuse model_state.pt if present; else train. Then evaluate and save."""
    if done(name, seed):
        log(f"skip {name} seed {seed}: complete")
        return
    spec = ARMS[name]
    tensors = BASE_TENSORS
    if spec.get("data") or spec.get("train_requests") is not None:
        train_arrays, val_arrays = arm_arrays(spec)
        tensors = {"train": M.SplitTensors(train_arrays, DEVICE), "val": BASE_TENSORS["val"],
                   "slates": BASE_TENSORS["slates"]}
        if val_arrays is not VAL:
            tensors.update(val=M.SplitTensors(val_arrays, DEVICE),
                           slates=M.SplitTensors(D.subset_split_arrays(val_arrays, SLATE_SAMPLE), DEVICE))
    model, cfg, acfg = arm_model(name, seed)
    out = arm_dir(name, seed)
    out.mkdir(parents=True, exist_ok=True)
    state = out / "model_state.pt"
    if state.is_file():
        model.load_state_dict(torch.load(state, map_location=DEVICE, weights_only=True))
        model.eval()
        train_history = json.loads((out / "train_history.json").read_text())
        log(f"{name} seed {seed}: reusing saved weights")
    else:
        tcfg = replace(TRAIN_CFG, seed=seed, max_epochs=EPOCHS)   # early-stopping fields are unused
        log(f"{name} seed {seed}: training {EPOCHS} epochs on {tensors['train'].n_impressions} requests")
        train_history = V.train_fixed_budget(model, tensors["train"], tcfg, EPOCHS, log)
        D.write_json({"arm": name, "seed": seed, "change": spec["change"], "model_config": cfg.to_json(),
                      "ablation": acfg.to_json(), "train_config": asdict(tcfg), "data": spec.get("data"),
                      "train_requests": tensors["train"].n_impressions}, out / "arm_config.json")
        D.write_json(train_history, out / "train_history.json")
        torch.save(model.state_dict(), state)              # written only after training completes
    cat, cat_per, slate, slate_per, timing = evaluate_model(model, tensors["val"], tensors["slates"])
    metrics = {"tier": spec["tier"], "group": spec["group"], "change": spec["change"], "kind": "trained",
               "parameters": M.count_parameters(model)["trainable_parameters"],
               "train_seconds": float(sum(h["seconds"] for h in train_history)),
               "train_history": train_history, "catalogue": cat, "recorded_slates": slate, "timing": timing}
    if name == "full" and seed == MAIN_SEED:
        metrics["reproducibility_check"] = selection_check(model)
        log("Reproducibility check:", metrics["reproducibility_check"])
    save_run(name, seed, metrics, cat_per, slate_per)
    del model, tensors
    free_gpu()

def comparator_metrics(name, catalogue, slate, timing, parameters, train_seconds=0.0, **extra):
    spec = COMPARATORS[name]
    return {"tier": spec["tier"], "group": spec["group"], "change": spec["change"], "kind": "no training",
            "parameters": parameters, "train_seconds": train_seconds, "catalogue": catalogue,
            "recorded_slates": slate, "timing": timing, **extra}

def run_ranker(name, ranker, **extra):
    """No-model comparator scored on both samples (``RecencyPopularityRanker``)."""
    if done(name, MAIN_SEED):
        log(f"skip {name}: complete")
        return
    cat, cat_per = V.evaluate_catalogue(lambda i, k: ranker.topk(int(VAL.imp_time_us[i]), k),
                                        CAT_SAMPLE, CLICKS_VAL, K)
    t0 = time.time()
    scores = ranker.slate_scores(SLATE_ARRAYS.cand_article_row,
                                 np.repeat(SLATE_ARRAYS.imp_time_us, np.diff(SLATE_ARRAYS.offsets)))
    slate, slate_per = V.slate_metrics(SLATE_ARRAYS.offsets, SLATE_ARRAYS.labels, scores)
    timing = {**latency(cat), "slate_scoring_seconds": time.time() - t0}
    save_run(name, MAIN_SEED, comparator_metrics(name, cat, slate, timing, 0, **extra), cat_per, slate_per)

## 2. Recency and time-aware popularity

Newest-first ranks eligible articles by publication time, breaking ties by
article ID. The popularity baseline ranks by clicks in the preceding 24 hours,
then publication time. It uses earlier training and validation events strictly
before each request, including no click from that request or a future request.
This is an updating popularity baseline; the neural model uses fixed histories.
Neither comparator requires model fitting.

In [ ]:
POPULARITY_WINDOW_HOURS = 24
run_ranker("most_recent", V.RecencyPopularityRanker(INDEX, ARTICLES))
if not done("popularity_24h", MAIN_SEED):
    click_t, click_ids, click_info = V.load_click_events([PATHS["behaviors_train"], PATHS["behaviors_validation"]])
    run_ranker("popularity_24h", V.RecencyPopularityRanker(INDEX, ARTICLES, click_t, click_ids, POPULARITY_WINDOW_HOURS),
               click_sources=click_info, window_hours=POPULARITY_WINDOW_HOURS)

most_recent seed 42 saved | Recall@100 0.8870666666666666 | slate AUC 0.5077564708974969
popularity_24h seed 42 saved | Recall@100 0.5279 | slate AUC 0.7220622037846953


## 3. Saved model and recency candidate subsets

Evaluate the initial run's saved model on the shared catalogue sample. Then
restrict its input to the newest 200, 500 or 1,000 eligible articles and retain
the 100 with highest neural scores. Report the corresponding recency recall
ceiling alongside each result.

These are stage-one retrieval comparisons. They do not implement the separate
business reranker. No model is retrained in this section.

In [ ]:
RERANK_N = (200, 500, 1000)
main_model = load_main_model()
if not done("main_model", MAIN_SEED):
    cat, cat_per, slate, slate_per, timing = evaluate_model(main_model, BASE_TENSORS["val"], BASE_TENSORS["slates"])
    save_run("main_model", MAIN_SEED, comparator_metrics(
        "main_model", cat, slate, timing, M.count_parameters(main_model)["trainable_parameters"],
        source=str(MAIN_RUN_DIR / "model")), cat_per, slate_per)
recency = V.RecencyPopularityRanker(INDEX, ARTICLES)
scorer = M.CatalogueScorer(main_model, BASE_TENSORS["val"], INDEX, AGE_STATS, chunk=CATALOGUE_CHUNK, requests=CAT_SAMPLE)
for n in RERANK_N:
    name = f"rerank_{n}"
    if done(name, MAIN_SEED):
        log(f"skip {name}: complete")
        continue
    ceiling, _ = V.evaluate_catalogue(lambda i, k: recency.topk(int(VAL.imp_time_us[i]), k), CAT_SAMPLE, CLICKS_VAL, n)
    cat, cat_per = V.evaluate_catalogue(V.RerankTopK(scorer, recency, INDEX, ARTICLES, n).topk,
                                        CAT_SAMPLE, CLICKS_VAL, K, sync=cuda_sync)
    save_run(name, MAIN_SEED, comparator_metrics(
        name, cat, None, latency(cat), M.count_parameters(main_model)["trainable_parameters"],
        candidates=n, recency_ceiling_recall=ceiling[f"known_click_recall@{n}"]), cat_per)
del main_model, scorer
free_gpu()

main_model seed 42 saved | Recall@100 0.72835 | slate AUC 0.7096629472995915
rerank_200 seed 42 saved | Recall@100 0.78875 | slate AUC None
rerank_500 seed 42 saved | Recall@100 0.76005 | slate AUC None
rerank_1000 seed 42 saved | Recall@100 0.75405 | slate AUC None


## 4. Full model across three seeds

Fit the complete architecture with seeds 42, 43 and 44 under the common
four-epoch budget. Their variation measures sensitivity to training randomness.
The seed-42 selection-sample check is descriptive; GPU nondeterminism means the
fit is not claimed to reproduce the initial weights exactly.

In [ ]:
for seed in SEEDS:
    run_trained_arm("full", seed)

full seed 42: training 4 epochs on 232887 requests
  epoch 1: train BCE 0.2853 (23s)
  epoch 2: train BCE 0.2745 (21s)
  epoch 3: train BCE 0.2708 (21s)
  epoch 4: train BCE 0.2683 (22s)
Reproducibility check: {'sample': 'main-run 100-request selection sample (exclusion set; not a reported result)', 'recall@100': 0.73, 'main_run_epoch4_recall@100': 0.735}
full seed 42 saved | Recall@100 0.72565 | slate AUC 0.7126933951248416
full seed 43: training 4 epochs on 232887 requests
  epoch 1: train BCE 0.2846 (22s)
  epoch 2: train BCE 0.2745 (23s)
  epoch 3: train BCE 0.2709 (23s)
  epoch 4: train BCE 0.2685 (23s)
full seed 43 saved | Recall@100 0.6709333333333333 | slate AUC 0.7084800847546954
full seed 44: training 4 epochs on 232887 requests
  epoch 1: train BCE 0.2850 (22s)
  epoch 2: train BCE 0.2747 (21s)
  epoch 3: train BCE 0.2708 (22s)
  epoch 4: train BCE 0.2684 (21s)
full seed 44 saved | Recall@100 0.60265 | slate AUC 0.7143129120306057


## 5. Attention and recurrence controls

Train each control with the same three seeds and training budget:

- `last_state` uses the final LSTM state instead of target-aware attention.
- `attention_no_lstm` attends to projected history-event inputs without recurrence.

Compare each control with the seed-matched full model to assess the contribution
of candidate-specific aggregation and recurrent history encoding.

In [ ]:
for name in ("last_state", "attention_no_lstm"):
    for seed in SEEDS:
        run_trained_arm(name, seed)

last_state seed 42: training 4 epochs on 232887 requests
  epoch 1: train BCE 0.2862 (20s)
  epoch 2: train BCE 0.2751 (19s)
  epoch 3: train BCE 0.2716 (19s)
  epoch 4: train BCE 0.2695 (19s)
last_state seed 42 saved | Recall@100 0.6731 | slate AUC 0.7058932182587571
last_state seed 43: training 4 epochs on 232887 requests
  epoch 1: train BCE 0.2860 (19s)
  epoch 2: train BCE 0.2759 (19s)
  epoch 3: train BCE 0.2720 (19s)
  epoch 4: train BCE 0.2698 (19s)
last_state seed 43 saved | Recall@100 0.6721333333333332 | slate AUC 0.7030167938472155
last_state seed 44: training 4 epochs on 232887 requests
  epoch 1: train BCE 0.2865 (19s)
  epoch 2: train BCE 0.2761 (19s)
  epoch 3: train BCE 0.2719 (19s)
  epoch 4: train BCE 0.2694 (19s)
last_state seed 44 saved | Recall@100 0.5895333333333334 | slate AUC 0.7098176224754549
attention_no_lstm seed 42: training 4 epochs on 232887 requests
  epoch 1: train BCE 0.2844 (14s)
  epoch 2: train BCE 0.2745 (13s)
  epoch 3: train BCE 0.2709 (13s)
  e

## 6. Metadata-only control

Remove both vector representations and their availability indicators, retaining
article metadata and the history-based model. The availability table describes
how articles with and without supplied vectors differ. Because the control
removes content and availability information together, its result does not
identify either factor's individual effect.

In [ ]:
real = np.arange(1, len(ARTICLES.article_ids))
has_vector = ARTICLES.has_text[real] | ARTICLES.has_image[real]
col = {n: i for i, n in enumerate(NUMERIC_NAMES)}
def availability_row(mask):
    rows = real[mask]
    return {"articles": int(len(rows)),
            "category unknown": float((ARTICLES.category[rows] == 2).mean()),
            "any topic unknown": float((ARTICLES.topics[rows] == 1).any(1).mean()),
            "no topics": float((ARTICLES.topics[rows] == 0).all(1).mean()),
            "body words missing": float(ARTICLES.numeric[rows, col["body_words_missing"]].mean()),
            "sentiment missing": float(ARTICLES.numeric[rows, col["sentiment_missing"]].mean()),
            "premium missing": float(ARTICLES.numeric[rows, col["premium_missing"]].mean()),
            "in_catalogue values": str(np.unique(ARTICLES.numeric[rows, col["in_catalogue"]]).tolist()),
            "median publication": str(pd.Timestamp(int(np.median(ARTICLES.published_s[rows])), unit="s").date())}
display(pd.DataFrame({"with a vector": availability_row(has_vector), "vector-less": availability_row(~has_vector)}).T)
run_trained_arm("metadata_only", MAIN_SEED)

,articles,category unknown,any topic unknown,no topics,body words missing,sentiment missing,premium missing,in_catalogue values,median publication
with a vector,21015,0.000619,0.0,0.009041,0.073519,0.0,0.0,[1.0],2023-01-31
vector-less,104526,0.000373,0.0,0.002392,0.039349,0.0,0.0,[1.0],2019-03-17


metadata_only seed 42: training 4 epochs on 232887 requests
  epoch 1: train BCE 0.2873 (20s)
  epoch 2: train BCE 0.2774 (21s)
  epoch 3: train BCE 0.2736 (20s)
  epoch 4: train BCE 0.2713 (20s)
metadata_only seed 42 saved | Recall@100 0.4041666666666667 | slate AUC 0.6979464711395144


## 7. No-history control

Remove history events and history-derived request context while retaining
candidate and remaining request inputs. This comparison tests the contribution
of historical user information under the fixed training budget.

In [ ]:
run_trained_arm("no_history", MAIN_SEED)

no_history seed 42: training 4 epochs on 232887 requests
  epoch 1: train BCE 0.2909 (7s)
  epoch 2: train BCE 0.2834 (7s)
  epoch 3: train BCE 0.2812 (7s)
  epoch 4: train BCE 0.2800 (7s)
no_history seed 42 saved | Recall@100 0.4638 | slate AUC 0.6647437895969845


## 8. Logistic regression on recorded slates

Fit a simpler click-prediction reference using training data and evaluate it on
the recorded-slate sample. This implementation reports slate metrics only;
its AUC must not be interpreted as catalogue Recall@100.

In [ ]:
if not done("logistic_regression", MAIN_SEED):
    t0 = time.time()
    x_train = V.lr_features(FRAMES["train"], ARTICLES, V.history_profiles(PATHS["history_train"], ARTICLES),
                            PREP["device_columns"])
    lr_model = V.make_lr_pipeline().fit(x_train, FRAMES["train"]["clicked"].to_numpy().astype(int))
    fit_seconds = time.time() - t0
    del x_train
    slate_rows, _ = D.subset_requests_frame(FRAMES["validation"], OFFSETS["validation"], SLATE_SAMPLE)
    x_val = V.lr_features(slate_rows, ARTICLES, V.history_profiles(PATHS["history_validation"], ARTICLES),
                          PREP["device_columns"])
    t0 = time.time()
    slate, slate_per = V.slate_metrics(SLATE_ARRAYS.offsets, SLATE_ARRAYS.labels, lr_model.decision_function(x_val))
    estimator = lr_model.named_steps["model"]
    save_run("logistic_regression", MAIN_SEED, comparator_metrics(
        "logistic_regression", None, slate, {"slate_scoring_seconds": time.time() - t0},
        int(estimator.coef_.size + estimator.intercept_.size), fit_seconds,
        scope="recorded-slate-only comparison: engineered features exist only for slate candidates",
        features={"numeric": V.LR_NUMERIC, "categorical": V.LR_CATEGORICAL},
        solver_iterations=int(estimator.n_iter_[0]), train_rows=int(len(FRAMES["train"]))), slates=slate_per)

logistic_regression seed 42 saved | Recall@100 None | slate AUC 0.7048490845673211


## 9. Evaluation-time history permutation

Apply a fixed permutation to history events, keeping their event fields aligned,
and score with the initial model's unchanged weights. This is a sensitivity
analysis, not a separately trained order-free model. A small change does not
establish that chronology is useful or unnecessary in general.

In [ ]:
PERMUTE_SEED = 0
if not done("permute_eval", MAIN_SEED):
    main_model = load_main_model()
    permuted = D.permute_history(VAL, PERMUTE_SEED)
    cat, cat_per, slate, slate_per, timing = evaluate_model(
        main_model, M.SplitTensors(permuted, DEVICE),
        M.SplitTensors(D.subset_split_arrays(permuted, SLATE_SAMPLE), DEVICE))
    save_run("permute_eval", MAIN_SEED, comparator_metrics(
        "permute_eval", cat, slate, timing, M.count_parameters(main_model)["trainable_parameters"],
        permute_seed=PERMUTE_SEED), cat_per, slate_per)

permute_eval seed 42 saved | Recall@100 0.73255 | slate AUC 0.709810753430226


## 10. Text and image controls

Train the model without the text-vector block, then without the image-vector
block. Each control retains its corresponding availability indicator, uses seed
42 and receives the common four-epoch budget.

In [ ]:
for name in ("no_text", "no_image"):
    run_trained_arm(name, MAIN_SEED)

no_text seed 42: training 4 epochs on 232887 requests
  epoch 1: train BCE 0.2865 (21s)
  epoch 2: train BCE 0.2759 (21s)
  epoch 3: train BCE 0.2721 (21s)
  epoch 4: train BCE 0.2697 (21s)
no_text seed 42 saved | Recall@100 0.5877166666666666 | slate AUC 0.6915009610744381
no_image seed 42: training 4 epochs on 232887 requests
  epoch 1: train BCE 0.2859 (21s)
  epoch 2: train BCE 0.2751 (21s)
  epoch 3: train BCE 0.2714 (21s)
  epoch 4: train BCE 0.2692 (21s)
no_image seed 42 saved | Recall@100 0.6211666666666666 | slate AUC 0.7110556230401224


## 11. Seed ensemble

Average the logits of the three full-model fits before selecting the top 100.
The ensemble uses existing fitted weights. Its retrieval benefit is considered
alongside the cost of scoring with three models.

In [ ]:
if not done("ensemble", ENSEMBLE_LABEL):
    for seed in SEEDS:
        run_trained_arm("full", seed)
    members = [load_arm_model("full", s) for s in SEEDS]
    cat, cat_per, slate, slate_per, timing = evaluate_model(None, BASE_TENSORS["val"], BASE_TENSORS["slates"],
                                                            members=members)
    save_run("ensemble", ENSEMBLE_LABEL, comparator_metrics(
        "ensemble", cat, slate, timing, int(sum(M.count_parameters(m)["trainable_parameters"] for m in members)),
        score="mean logit over seeds " + ENSEMBLE_LABEL), cat_per, slate_per)

skip full seed 42: complete
skip full seed 43: complete
skip full seed 44: complete
ensemble seed 42_43_44 saved | Recall@100 0.71965 | slate AUC 0.7186001756569174


## 12. Neural score and freshness blend

The blended score is the mean neural logit minus
`lambda * log1p(article age in hours)`. At zero weight it is the neural ensemble;
a dominant freshness term approaches newest-first.

Select the weight from `{0, 0.25, 0.5, 1, 2, 4, 8, 16, 64}` using only the
reserved 2,000-request sample: maximise Recall@100, breaking ties by nDCG@100.
Slate AUC is descriptive and does not select the weight. Report the chosen
weight on the separate 5,000-request catalogue sample for the ensemble and each
single-seed model. No neural weights are refitted.

The section retains the original additional decision rule: the paired 95%
interval must exclude zero and the absolute gain must exceed the three
single-seed blends' Recall@100 spread. Interpret that spread as a robustness
heuristic rather than a formal significance threshold.

In [ ]:
BLEND_LAMBDAS = (0, 0.25, 0.5, 1, 2, 4, 8, 16, 64)
COMPARATORS["ensemble_recency"] = dict(tier=1, group="Seeds", seeds=[ENSEMBLE_LABEL], reference="most_recent",
                                       change="mean logit of three seeds - lambda * log1p(age hours)")
COMPARATORS["full_recency"] = dict(tier=1, group="Seeds", seeds=list(SEEDS), reference="most_recent",
                                   change="single seed logit - lambda * log1p(age hours)")

@torch.no_grad()
def blend_tops(scorers, requests, lambdas, k):
    """Exact top-k of mean logit - lambda * log1p(age_h) for every lambda, one pass per request."""
    s0, out = scorers[0], {lam: {} for lam in lambdas}
    for i in map(int, requests):
        t, pos = s0.data.imp_time[i], int(s0.lo[i])
        best = {lam: (torch.empty(0, dtype=torch.long, device=s0.device),
                      torch.empty(0, dtype=torch.float32, device=s0.device)) for lam in lambdas}
        for parts in zip(*(s.iter_score_chunks(i) for s in scorers)):
            ids = parts[0][0]
            mean = torch.stack([p[1].float() for p in parts]).mean(0)
            age_h = ((t - s0.pub[pos:pos + len(ids)]).double() / 3600.0).clamp(min=0)
            fresh = -torch.log1p(age_h).float()
            pos += len(ids)
            for lam in lambdas:
                top_ids, top_scores = best[lam]
                best[lam] = M.select_top_k(torch.cat([top_scores, mean + lam * fresh]),
                                           torch.cat([top_ids, ids]), k)
        for lam in lambdas:
            out[lam][i] = (best[lam][0].cpu(), best[lam][1].cpu())
    return out

def blend_slate(logits, arrays, lam):
    counts = np.diff(arrays.offsets)
    age_h = np.clip((np.repeat(arrays.imp_time_us, counts) / 1e6
                     - ARTICLES.published_s[arrays.cand_article_row].astype(np.float64)) / 3600.0, 0, None)
    return V.slate_metrics(arrays.offsets, arrays.labels, logits - lam * np.log1p(age_h))

members = [load_arm_model("full", s) for s in SEEDS]
scorers = [M.CatalogueScorer(m, BASE_TENSORS["val"], INDEX, AGE_STATS, chunk=CATALOGUE_CHUNK,
                             requests=np.union1d(EXCLUSION, CAT_SAMPLE)) for m in members]
batch = TRAIN_CFG.eval_batch_impressions

# Selection: 2,000-request exclusion sample only
select_path = ABLATION_DIR / "recency_blend_selection.json"
if select_path.is_file():
    selection = json.loads(select_path.read_text())
else:
    t0 = time.time()
    tops = blend_tops(scorers, EXCLUSION, BLEND_LAMBDAS, K)
    ex_arrays = D.subset_split_arrays(VAL, EXCLUSION)
    ex_tensors = M.SplitTensors(ex_arrays, DEVICE)
    ex_logits = np.mean([M.predict_logits(m, ex_tensors, batch_size=batch) for m in members], axis=0)
    curve = []
    for lam in BLEND_LAMBDAS:
        cat, _ = V.evaluate_catalogue(lambda i, k, lam=lam: tops[lam][i], EXCLUSION, CLICKS_VAL, K)
        slate, _ = blend_slate(ex_logits, ex_arrays, lam)
        curve.append({"lambda": lam, "Recall@100": cat["known_click_recall@100"],
                      "nDCG@100": cat["observed_click_ndcg@100"],
                      "slate AUC": slate["mean_impression_auc"], "slate nDCG@5": slate["mean_impression_ndcg@5"]})
    chosen = max(curve, key=lambda r: (r["Recall@100"], r["nDCG@100"]))["lambda"]
    selection = {"chosen_lambda": chosen, "curve": curve, "sample": "2,000-request exclusion sample",
                 "seconds": time.time() - t0}
    D.write_json(selection, select_path)
    del ex_tensors, tops
print(pd.DataFrame(selection["curve"]).round(4).to_string(index=False))
LAM = selection["chosen_lambda"]
log("Chosen lambda:", LAM)

# Report: 5,000-request catalogue sample and recorded slates
def report_blend(name, seed, group, models):
    if done(name, seed):
        log(f"skip {name} seed {seed}: complete")
        return
    t0 = time.time()
    tops = blend_tops(group, CAT_SAMPLE, (LAM,), K)[LAM]
    seconds = time.time() - t0
    cat, cat_per = V.evaluate_catalogue(lambda i, k: tops[i], CAT_SAMPLE, CLICKS_VAL, K)
    logits = np.mean([M.predict_logits(m, BASE_TENSORS["slates"], batch_size=batch) for m in models], axis=0)
    slate, slate_per = blend_slate(logits, SLATE_ARRAYS, LAM)
    timing = {"catalogue_ms_per_request": 1e3 * seconds / len(CAT_SAMPLE),
              "catalogue_ms_per_1k_candidates": 1e6 * seconds / CAT_PAIRS}
    params = int(sum(M.count_parameters(m)["trainable_parameters"] for m in models))
    save_run(name, seed, comparator_metrics(name, cat, slate, timing, params, blend_lambda=LAM), cat_per, slate_per)

for s, scorer, member in zip(SEEDS, scorers, members):
    report_blend("full_recency", s, [scorer], [member])
report_blend("ensemble_recency", ENSEMBLE_LABEL, scorers, members)

recall = lambda name, seed: pd.read_parquet(arm_dir(name, seed) / "catalogue.parquet")["recall@100"].to_numpy()
auc = lambda name, seed: pd.read_parquet(arm_dir(name, seed) / "slates.parquet")["auc"].to_numpy()
single = {s: float(np.nanmean(recall("full_recency", s))) for s in SEEDS}
floor = max(single.values()) - min(single.values())
log("Single-seed blend Recall@100:", single, "| noise floor:", round(floor, 4))
for ref, seed in (("most_recent", MAIN_SEED), ("ensemble", ENSEMBLE_LABEL)):
    d = V.paired_delta(CAT_USERS, [recall("ensemble_recency", ENSEMBLE_LABEL)], [recall(ref, seed)], N_BOOT, BOOT_SEED)
    a = V.paired_delta(SLATE_USERS, [auc("ensemble_recency", ENSEMBLE_LABEL)], [auc(ref, seed)], N_BOOT, BOOT_SEED)
    log(f"blend vs {ref}: Recall@100 {fmt(d) if 'fmt' in globals() else d} | beats floor: {V.differs(d, floor)}"
        f" | slate AUC {fmt(a) if 'fmt' in globals() else a}")
del members, scorers
free_gpu()

 lambda  Recall@100  nDCG@100  slate AUC  slate nDCG@5
   0.00      0.7039    0.1972     0.7220        0.5536
   0.25      0.8244    0.2541     0.7087        0.5405
   0.50      0.8644    0.2777     0.6934        0.5201
   1.00      0.8859    0.2843     0.6618        0.4883
   2.00      0.8919    0.2723     0.6173        0.4493
   4.00      0.8899    0.2554     0.5737        0.4095
   8.00      0.8859    0.2464     0.5465        0.3842
  16.00      0.8834    0.2408     0.5304        0.3711
  64.00      0.8804    0.2367     0.5213        0.3623
Chosen lambda: 2
full_recency seed 42 saved | Recall@100 0.8952666666666667 | slate AUC 0.5996518822617718
full_recency seed 43 saved | Recall@100 0.8972666666666667 | slate AUC 0.6093726849467839
full_recency seed 44 saved | Recall@100 0.8857333333333333 | slate AUC 0.6088932499102122
ensemble_recency seed 42_43_44 saved | Recall@100 0.8970666666666666 | slate AUC 0.6074544729413579
Single-seed blend Recall@100: {42: 0.8952666666666667, 43: 0.89

## 13. Saved results summary

Reload all completed experiment metrics from Drive, including the freshness
blends. Each row averages the available seeds listed in that row; the saved
initial model remains a separate single-run reference. Missing catalogue
metrics for logistic regression mean that catalogue retrieval was not evaluated.

This section is self-contained and can run in a fresh CPU runtime. It does not
fit models or generate predictions. The compact table reports seed averages;
paired intervals remain in the saved comparison outputs and segment report.

In [5]:
from google.colab import drive
from pathlib import Path
import json
import pandas as pd

drive.mount("/content/drive")

root = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "BT4222 Shared Final Folder/Experiments/2.1_lstm/"
    "ablation/ablation_v1/arms"
)

paths = sorted(root.glob("*/metrics.json"))
if not paths:
    raise FileNotFoundError(f"No saved results found under: {root}")

rows = []
for path in paths:
    metrics = json.loads(path.read_text())
    model, seed = path.parent.name.rsplit("_seed", 1)
    catalogue = metrics.get("catalogue") or {}
    slates = metrics.get("recorded_slates") or {}
    timing = metrics.get("timing") or {}

    rows.append({
        "model": model,
        "seeds": seed,
        "Recall@100": catalogue.get("known_click_recall@100"),
        "nDCG@100": catalogue.get("observed_click_ndcg@100"),
        "slate AUC": slates.get("mean_impression_auc"),
        "ms / request": timing.get("catalogue_ms_per_request"),
    })

RESULTS = (
    pd.DataFrame(rows)
    .groupby("model", as_index=False)
    .agg({
        "seeds": lambda values: ", ".join(values),
        "Recall@100": "mean",
        "nDCG@100": "mean",
        "slate AUC": "mean",
        "ms / request": "mean",
    })
    .sort_values("Recall@100", ascending=False)
    .reset_index(drop=True)
)

print(f"Loaded {len(paths)} saved runs. Multiple seeds are averaged.")
display(RESULTS.round(4))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 26 saved runs. Multiple seeds are averaged.


,model,seeds,Recall@100,nDCG@100,slate AUC,ms / request
0,ensemble_recency,42_43_44,0.8971,0.2726,0.6075,33.5415
1,full_recency,"42, 43, 44",0.8928,0.2695,0.6060,12.9717
2,most_recent,42,0.8871,0.2356,0.5078,0.2465
3,rerank_200,42,0.7887,0.2125,NaN,2.1894
4,rerank_500,42,0.7600,0.2049,NaN,2.2768
5,rerank_1000,42,0.7540,0.2017,NaN,2.4262
6,permute_eval,42,0.7326,0.1982,0.7098,12.3747
7,main_model,42,0.7284,0.1965,0.7097,12.3594
8,ensemble,42_43_44,0.7196,0.2023,0.7186,32.9128
9,full,"42, 43, 44",0.6664,0.1780,0.7118,12.3887


## 14. Saved segment comparisons

Reload the segment report produced by the completed analysis. It compares
per-request retrieval differences by complete-history length and by whether
clicked articles were referenced in training candidates or histories. An unseen
training article is not necessarily newly published. The model still receives
at most 50 history events.

The report uses paired user-bootstrap intervals. There were no empty-history
requests in this sample, so no empirical result is available for that segment.
Run Section 13 first in a fresh runtime; no model setup is required.

In [ ]:
SEGMENTS = pd.DataFrame(json.loads((root.parent / "segment_table.json").read_text()))
display(SEGMENTS)

/content/ablation_variants.py:641: RuntimeWarning: Mean of empty slice
  a = np.nanmean(np.vstack(arm_runs), axis=0) if len(arm_runs) > 1 else np.asarray(arm_runs[0])
/content/ablation_variants.py:642: RuntimeWarning: Mean of empty slice
  b = np.nanmean(np.vstack(ref_runs), axis=0) if len(ref_runs) > 1 else np.asarray(ref_runs[0])


,segment,requests,full Recall@100,most_recent vs full,rerank_500 vs most_recent,last_state vs full,attention_no_lstm vs full,metadata_only vs full,no_history vs full,permute_eval vs main_model,no_text vs full,no_image vs full,popularity_24h vs full
0,history 0 (empty),0,NaN,None,None,None,None,None,None,None,None,None,None
1,history 1-9,71,0.553991,"+0.3099 [+0.1857, +0.4203]","-0.1972 [-0.2836, -0.1111]","+0.0610 [+0.0046, +0.1257]","-0.0376 [-0.1142, +0.0326]","-0.1268 [-0.2500, -0.0263]","-0.1408 [-0.2833, -0.0169]","+0.0141 [-0.0339, +0.0595]","-0.0845 [-0.1970, +0.0143]","-0.0141 [-0.0893, +0.0595]","-0.0141 [-0.1692, +0.1085]"
2,history 10-19,155,0.619355,"+0.1935 [+0.1190, +0.2677]","-0.1677 [-0.2390, -0.1027]","-0.0086 [-0.0488, +0.0290]","-0.1591 [-0.2061, -0.1181]","-0.2194 [-0.2985, -0.1465]","-0.1677 [-0.2696, -0.0872]","+0.0194 [-0.0084, +0.0546]","-0.0839 [-0.1592, -0.0244]","-0.0774 [-0.1488, -0.0121]","-0.1226 [-0.2326, -0.0389]"
3,history 20-49,442,0.627828,"+0.1580 [+0.1049, +0.2101]","-0.1297 [-0.1795, -0.0797]","-0.0177 [-0.0380, +0.0023]","-0.1385 [-0.1653, -0.1130]","-0.2764 [-0.3219, -0.2287]","-0.2240 [-0.2783, -0.1691]","+0.0068 [-0.0083, +0.0219]","-0.1105 [-0.1588, -0.0637]","-0.0845 [-0.1235, -0.0476]","-0.1949 [-0.2487, -0.1433]"
4,history 50-99,681,0.687469,"+0.1329 [+0.0987, +0.1664]","-0.1094 [-0.1394, -0.0791]","-0.0233 [-0.0391, -0.0091]","-0.1378 [-0.1609, -0.1168]","-0.2599 [-0.3024, -0.2161]","-0.2944 [-0.3328, -0.2531]","-0.0044 [-0.0149, +0.0073]","-0.1065 [-0.1418, -0.0716]","-0.0844 [-0.1111, -0.0596]","-0.2078 [-0.2621, -0.1586]"
5,history 100-199,1093,0.662092,"+0.1532 [+0.1249, +0.1821]","-0.1139 [-0.1399, -0.0870]","-0.0171 [-0.0305, -0.0042]","-0.1287 [-0.1446, -0.1135]","-0.2969 [-0.3283, -0.2659]","-0.2420 [-0.2756, -0.2109]","+0.0073 [-0.0009, +0.0162]","-0.1272 [-0.1532, -0.1034]","-0.0970 [-0.1214, -0.0747]","-0.2338 [-0.2717, -0.1960]"
6,history 200+,2558,0.675289,"+0.1670 [+0.1488, +0.1854]","-0.1324 [-0.1504, -0.1140]","-0.0266 [-0.0371, -0.0173]","-0.1547 [-0.1660, -0.1437]","-0.3678 [-0.3901, -0.3462]","-0.2773 [-0.2996, -0.2561]","+0.0035 [-0.0023, +0.0094]","-0.1604 [-0.1822, -0.1395]","-0.1206 [-0.1393, -0.1036]","-0.1898 [-0.2178, -0.1607]"
7,cold clicked article,4754,0.687573,"+0.1748 [+0.1604, +0.1885]","-0.1382 [-0.1510, -0.1249]","-0.0218 [-0.0289, -0.0147]","-0.1485 [-0.1570, -0.1402]","-0.3320 [-0.3494, -0.3141]","-0.2737 [-0.2909, -0.2579]","+0.0044 [+0.0000, +0.0088]","-0.1398 [-0.1548, -0.1249]","-0.1088 [-0.1210, -0.0970]","-0.2088 [-0.2293, -0.1886]"
8,warm clicked article,246,0.257453,"-0.0976 [-0.1548, -0.0449]","+0.0894 [+0.0381, +0.1411]","-0.0149 [-0.0372, +0.0068]","-0.0515 [-0.0736, -0.0317]","-0.1179 [-0.1600, -0.0790]","-0.0325 [-0.0769, +0.0083]","+0.0000 [+0.0000, +0.0000]","-0.1016 [-0.1471, -0.0600]","-0.0203 [-0.0472, +0.0042]","+0.0163 [-0.0342, +0.0625]"


## 15. Saved vector-coverage comparisons

Reload the coverage report produced by the completed analysis. It compares the
share of articles without either supplied vector in the full eligible pool,
the recent pool, observed clicks and each model's top 100. The recent pool is
limited to articles published in the preceding 72 hours for this diagnostic
only; this does not change the shared catalogue eligibility policy.

Coverage differences describe the data and retrieved lists. They do not by
themselves establish leakage, fairness or relevance for unexposed articles.
Run Section 13 first in a fresh runtime; no scoring is required.

In [ ]:
VECTOR_SHARE = pd.DataFrame(json.loads((root.parent / "vector_share_table.json").read_text()))
display(VECTOR_SHARE.round(4))

,ranking,vector-less share,ratio to fresh pool
0,eligible pool (all),0.8415,3.3076
1,fresh pool (<= 72 h; median size 463),0.2544,1.0000
2,known clicks,0.0000,0.0000
3,most_recent top 100,0.1941,0.7630
4,popularity_24h top 100,0.0000,0.0000
5,main_model top 100,0.1412,0.5551
6,rerank_200 top 100,0.1827,0.7179
7,rerank_500 top 100,0.1539,0.6048
8,rerank_1000 top 100,0.1494,0.5872
9,full top 100,0.2556,1.0048


## 16. Findings and implications

On the same 5,000 validation requests, newest-first recovered 88.71% of known
clicks at 100, compared with 72.84% for the saved initial LSTM. The freshness
blend selected weight 2 on the separate 2,000-request sample and recovered
89.71% on the reporting sample. Its gain over newest-first was 1.00 percentage
point, with a paired 95% interval of approximately 0.60–1.44 points. It did not
pass the study's additional single-seed-spread rule.

Under the four-epoch budget, mean Recall@100 was 66.64% for the full model,
64.49% without target-aware attention and 52.27% without the LSTM. Recurrence
therefore has stronger support in this comparison than attention. The full
model's 12.30-point seed spread makes the smaller architectural differences
uncertain. Evaluation-time history permutation produced little change, so the
study does not establish a benefit specifically attributable to event order.

The single-seed freshness blends recovered 88.57–89.73%, while the blended
ensemble recovered 89.71%. This motivates evaluating a single blended model's
cost and retrieval performance before selecting a three-model ensemble.
Configurations must be chosen on selection data, not the best reporting seed.

The next experiment will examine improved checkpoint selection and inference
efficiency. It remains separate from this completed fixed-budget study. The
business reranker will be evaluated after selecting the stage-one model. These
observational retrieval metrics do not demonstrate a change in online click
rate, revenue or retention.